In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

spectral_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "eigen.jl"
)

@assert isfile(spectral_path) "Copy spectral.jl into the LowLevelFEM src directory first: $spectral_path"

Base.include(
    LowLevelFEM,
    spectral_path
)

using LowLevelFEM: spectralTransformation

In [2]:
structured_rect_mesh(lx=10)

mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u);

In [3]:
ndofs(U)

2222

In [4]:
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U))
M = ∫(U ⋅ mat.ρ ⋅ U);

In [5]:
bc = BoundaryCondition("left", ux=0, uy=0)

BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [6]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), solver=:direct, output=:raw)
λ

6-element Vector{Float64}:
 -2.6879879807254734
 -2.327613366938404
  1.255336890529273
  9.976753958142934e10
  6.775378057879274e11
  2.264829897568584e12

In [7]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), solver=:arpack, output=:raw)
λ

6-element Vector{Float64}:
 1.734579465689908
 1.9414645770848866
 3.81599236317599
 9.976753958448192e10
 6.775378057873097e11
 2.2648298975711226e12

In [8]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), support=[bc], solver=:direct, output=:raw)
λ

6-element Vector{Float64}:
 2.603546177375141e9
 9.376097657421771e10
 6.302018177369926e11
 6.512032800501093e11
 2.1534328737267305e12
 5.016371047742017e12

In [9]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), shift=0, nev=4, solver=:arpack, output=:raw)
λ

4-element Vector{Float64}:
 9.976753988528217e10
 6.77537805923219e11
 2.2648299090669463e12
 2.5129047657661235e12

In [10]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), support=[bc], output=:raw)
λ

6-element Vector{Float64}:
 2.60354617950967e9
 9.37609765769679e10
 6.302018177365557e11
 6.512032800474111e11
 2.153432873727252e12
 5.016371047741772e12

In [11]:
λ, ϕ = solveEigen(
    Symmetric(K),
    Symmetric(M);
    support=[bc],
    n=6,
    shift=0.0,
    solver=:arpack,
    solveroptions=(ncv=40, maxiter=10000), output=:raw
)

λ

6-element Vector{Float64}:
 2.603546179428854e9
 9.376097657723297e10
 6.302018177386315e11
 6.512032800499182e11
 2.1534328737271055e12
 5.01637104774217e12

In [12]:
λ, ϕ = solveEigen(
    Symmetric(K),
    Symmetric(M);
    support=[bc],
    n=6,
    sigma=0.0,
    which=:LM,
    solver=:arpack,
    solveroptions=(
        explicittransform=:none,
        ncv=40,
        maxiter=10000
    ), output=:raw
)

@show λ
@show sqrt.(real.(λ)) ./ (2π)

λ = [2.6035461792842407e9, 9.376097657714377e10, 6.302018177385378e11, 6.512032800498848e11, 2.1534328737270857e12, 5.016371047742034e12]
sqrt.(real.(λ)) ./ (2π) = [8120.874037820151, 48733.90279430151, 126345.5520653789, 128433.53059698537, 233553.188621109, 356463.410515854]


6-element Vector{Float64}:
   8120.874037820151
  48733.90279430151
 126345.5520653789
 128433.53059698537
 233553.188621109
 356463.410515854

In [13]:
λ, ϕ = solveEigen(Symmetric(K), Symmetric(M), sigma=1, which=:LM, output=:raw)
λ

6-element Vector{Float64}:
 -0.19261029809210983
 -0.15806082920636633
 -0.09786844977501774
  9.976753958290143e10
  6.775378057853104e11
  2.264829897570394e12

In [14]:
λ, ϕ = solveEigen(K, M, support=[bc], output=:raw)
λ

6-element Vector{Float64}:
 2.6035461788994546e9
 9.376097657584633e10
 6.302018177375157e11
 6.512032800464766e11
 2.153432873726888e12
 5.016371047741866e12

In [15]:
P = LowLevelFEM.prepare_eigen_system(
    K, M;
    support=[bc]
)

Kr, Mr = P.A, P.B

@show size(Kr)
@show nnz(Kr), nnz(Mr)

@show norm(Kr - Kr')
@show norm(Mr - Mr')

@show extrema(diag(Kr))
@show extrema(diag(Mr))

@show isposdef(Symmetric(Kr))
@show isposdef(Symmetric(Mr))

size(Kr) = (2200, 2200)
(nnz(Kr), nnz(Mr)) = (36815, 18476)
norm(Kr - Kr') = 7.624083117896295e-10
norm(Mr - Mr') = 0.0
extrema(diag(Kr)) = (98901.09890109849, 395604.39560440095)
extrema(diag(Mr)) = (8.722222222222265e-12, 3.488888888888941e-11)
isposdef(Symmetric(Kr)) = true
isposdef(Symmetric(Mr)) = true


true

In [16]:
per = MultiPointConstraint(master="left", slave="right")

MultiPointConstraint("left", "right", nothing, Dict{Symbol, Bool}())

In [17]:
e1 = VectorField(U, "left", [-1, 0, 0])
cs = CoordinateSystem(e1)

CoordinateSystem(VectorField([[-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [215, 216, 217, 218, 219, 220, 221, 222, 223, 224], 1, :v3D, Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1111, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), :field), nothing, 1.0)

In [18]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), coordSys=[cs], mpc=[per], solver=:direct, output=:raw)
λ

6-element Vector{Float64}:
 2.0129937711412113e10
 2.012993771605723e10
 1.3155147134091475e12
 1.3155147134114658e12
 2.5129049216648975e12
 2.5129049216735977e12

In [19]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), coordSys=[cs], mpc=[per], output=:raw)
λ

6-element Vector{Float64}:
 2.0129937713006264e10
 2.012993771389429e10
 1.3155147134137727e12
 1.315514713415211e12
 2.512904921666346e12
 2.512904921668472e12

In [20]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), coordSys=[cs], mpc=[per], solveroptions=(ncv=40, maxiter=10000), output=:raw)
λ

6-element Vector{Float64}:
 2.0129937712786022e10
 2.0129937730552788e10
 1.3155147134109226e12
 1.3155147134139116e12
 2.5129049216603037e12
 2.5129049216842485e12

In [ ]:
u = solveEigen(K, M)


XOpenIM() failed
Fontconfig warning: using without calling FcInit()


In [22]:
showDoFResults(u)
showEigenResults(u)
showModalResults(u)
openPostProcessor()